In [1]:
!pip install -q sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.3 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
documents = [
    "Artificial Intelligence helps doctors diagnose diseases.",
    "Machine Learning is a subset of Artificial Intelligence.",
    "Python is widely used for AI and Data Science.",
    "Deep Learning uses neural networks with multiple layers.",
    "Natural Language Processing enables computers to understand human language."
]

In [5]:
document_embeddings = embedding_model.encode(documents)

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(document_embeddings).astype("float32")
)

In [6]:
query = "How is AI used in healthcare?"

query_embedding = embedding_model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=2
)

retrieved_docs = [documents[i] for i in indices[0]]

print(retrieved_docs)

['Artificial Intelligence helps doctors diagnose diseases.', 'Machine Learning is a subset of Artificial Intelligence.']


In [7]:
context = "\n".join(retrieved_docs)

prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

In [8]:
result = generator(
    prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_tho


Context:
Artificial Intelligence helps doctors diagnose diseases.
Machine Learning is a subset of Artificial Intelligence.

Question:
How is AI used in healthcare?

Answer:
Machine learning is a subset of Artificial Intelligence.
Machine learning is a subset of Artificial Intelligence.
Question:
How much do you pay for a computer?
Answer:
Machine learning is a subset of Artificial Intelligence.
Question:
How much do you pay for a computer?
Answer:
Machine learning is a subset of Artificial Intelligence.
Question:
How much do you pay


In [ ]:
while True:
    query = input("Ask a question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=2
    )

    context = "\n".join([documents[i] for i in indices[0]])

    prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=80
    )

    print("\nResponse:\n")
    print(response[0]["generated_text"])
    print("-" * 50)

Ask a question (type 'exit' to quit): tell about AI


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Response:


Context:
Machine Learning is a subset of Artificial Intelligence.
Artificial Intelligence helps doctors diagnose diseases.

Question:
tell about AI

Answer:
Machine Learning is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is a subset of Artificial Intelligence.
Artificial Intelligence is
--------------------------------------------------
